In [150]:
import torch
from torch import nn
import torch.nn as nn
from torchvision import datasets,transforms
import torch.nn.functional as F
import math
import torch.optim as optim

In [151]:
# 1. Read the Tiny Shakespeare text from your file
with open('/input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [152]:
# 2. Build the character vocabulary from the real text
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [153]:
# 3. Create stoi and itos mappings
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


In [154]:
# 4. Tokenize the ENTIRE dataset (convert every character in Shakespeare to an integer)
data= torch.tensor([stoi[c] for c in text],dtype=torch.long)

In [155]:
# 5. Take a small chunk of it to create your input sequence tensor
#context window
block_size=8
x=data[:block_size]

# Add a batch dimension -> shape becomes (batch_size=1, sequence_length=8)
x=x.unsqueeze(0)
print(x)
print(x.shape)

tensor([[18, 47, 56, 57, 58,  1, 15, 47]])
torch.Size([1, 8])


In [156]:
# 6. Pass it through your embedding layer
embedding_dim=32
embedding_layer=nn.Embedding(vocab_size,embedding_dim)
embedding_output=embedding_layer(x)
print(embedding_output.shape)

torch.Size([1, 8, 32])


In [157]:
#7. initialize w,q,k

head_dim=32
Wq=nn.Linear(embedding_dim, head_dim, bias=False)
Wk=nn.Linear(embedding_dim, head_dim, bias=False)
Wv=nn.Linear(embedding_dim, head_dim, bias=False)

Q = Wq(embedding_output)
K = Wk(embedding_output)
V = Wv(embedding_output)

In [158]:
dropout=0.1

In [159]:
seq_len=Q.shape[1]
scaled_scores=Q @ K.transpose(2,1)/math.sqrt(embedding_dim)

In [160]:
casual_mask=torch.full((seq_len,seq_len),float('-inf'),device=scaled_scores.device)
casual_mask=torch.triu(casual_mask,diagonal=1)
masked_scores=scaled_scores+casual_mask
print(masked_scores)

tensor([[[-0.1396,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.1119,  0.1387,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [-0.5801,  0.8864, -0.7170,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.2499, -0.4488, -0.2174, -0.1460,    -inf,    -inf,    -inf,
             -inf],
         [ 0.1872, -0.2540,  0.2019, -0.1476, -0.2320,    -inf,    -inf,
             -inf],
         [ 0.5795,  0.1084,  0.5511,  0.2452,  0.5794,  0.3006,    -inf,
             -inf],
         [ 0.2847,  0.2699, -0.7944,  0.1108, -0.1015,  0.0044, -0.2570,
             -inf],
         [ 0.1119,  0.1387, -0.7033,  0.3495,  0.1188, -0.1128, -0.3376,
           0.1387]]], grad_fn=<AddBackward0>)


In [161]:
attention_weights=torch.softmax(masked_scores,dim=-1)
attention_weights_dropped=F.dropout(attention_weights,p=dropout,training=True)
scores=attention_weights_dropped@V
print(scores.shape)

torch.Size([1, 8, 32])


In [162]:
#verification
print(attention_weights[0, 0].detach().cpu().numpy())
print(attention_weights[0, 7].detach().cpu().numpy())

[1. 0. 0. 0. 0. 0. 0. 0.]
[0.13871676 0.14249362 0.06139155 0.17591836 0.13968702 0.11080107
 0.08849798 0.14249362]


In [163]:
#FFN
embedding_dim=32
hidden_dim=embedding_dim * 4
dropout=0.1

class FeedForward(nn.Module):
  def __init__(self,embedding_dim,hidden_dim,dropout):
    super().__init__()
    self.net=nn.Sequential(
        nn.Linear(embedding_dim,hidden_dim),
        nn.GELU(),
        nn.Linear(hidden_dim,embedding_dim),
        nn.Dropout(dropout),
    )
  def forward(self,x):
    return self.net(x)

ffn=FeedForward(embedding_dim,hidden_dim,dropout)
ffn_output=scores+ ffn(scores)
print(ffn_output.shape)

torch.Size([1, 8, 32])


In [164]:
# Map from embedding dimension (32) to total unique characters in vocabulary (65)
lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)
logits = lm_head(ffn_output)
print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 8, 65])


In [165]:
#loss
target_ids = data[1:block_size+1].unsqueeze(0)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits.view(-1, vocab_size), target_ids.view(-1))
print("Loss:", loss)

Loss: tensor(4.1086, grad_fn=<NllLossBackward0>)


#proper class way

In [166]:
block_size = 32
batch_size = 64
vocab_size = len(chars)

# 1. Split data into train and validation
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
val_data = data[split_idx:]

# 2. Helper to prepare sequential matrices for train and val
def prepare_data_matrix(dataset, batch_size, block_size):
    chunk_size = block_size * batch_size
    num_chunks = (len(dataset) - 1) // chunk_size
    truncated = dataset[:num_chunks * chunk_size + 1]

    x_matrix = truncated[:-1].view(batch_size, -1)
    y_matrix = truncated[1:].view(batch_size, -1)
    return x_matrix, y_matrix

train_x_mat, train_y_mat = prepare_data_matrix(train_data, batch_size, block_size)
val_x_mat, val_y_mat = prepare_data_matrix(val_data, batch_size, block_size)

# 3. Sequential batch getter
def get_sequential_batch(split, step_idx):
    # Correct reference:
    x_mat = train_x_mat if split == "train" else val_x_mat
    y_mat = train_y_mat if split == "train" else val_y_mat

    num_steps = x_mat.shape[1] // block_size
    col_idx = step_idx % num_steps

    start = col_idx * block_size
    end = start + block_size

    return x_mat[:, start:end], y_mat[:, start:end]

In [167]:
class RefinedShakespeareTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, block_size, dropout_p):
        super().__init__()
        # Token and Positional Embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)

        # Layer Normalization (Pre-LN architecture)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        # Q, K, V Projections
        self.Wq = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.Wk = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.Wv = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embedding_dim),
            nn.Dropout(dropout_p)
        )

        # Language Modeling Head
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.dropout_p = dropout_p
        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape

        # 1. Combine Token and Positional Embeddings
        tok_emb = self.embedding(idx) # [B, T, C]
        positions = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding(positions) # [T, C]
        emb = tok_emb + pos_emb # Broadcasted addition [B, T, C]

        # 2. Pre-LN Self-Attention Block with Residual Connection
        norm_emb = self.ln1(emb)
        Q = self.Wq(norm_emb)
        K = self.Wk(norm_emb)
        V = self.Wv(norm_emb)

        scaled_scores = (Q @ K.transpose(-2, -1)) / math.sqrt(emb.shape[-1])
        causal_mask = torch.full((T, T), float('-inf'), device=idx.device)
        causal_mask = torch.triu(causal_mask, diagonal=1)

        masked_scores = scaled_scores + causal_mask
        attn_weights = torch.softmax(masked_scores, dim=-1)
        attn_weights = F.dropout(attn_weights, p=self.dropout_p, training=self.training)

        attn_out = attn_weights @ V
        x = emb + attn_out # First Residual Connection

        # 3. Pre-LN Feed-Forward Block with Residual Connection
        x = x + self.ffn(self.ln2(x)) # Second Residual Connection

        # 4. Final Prediction Head
        logits = self.lm_head(x) # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # Autoregressive text generation helper
        for _ in range(max_new_tokens):
            # Crop context if it exceeds block_size
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)
            # Focus only on the very last time step
            logits = logits[:, -1, :] # [B, vocab_size]
            probs = torch.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append to running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Instantiate the refined model
model = RefinedShakespeareTransformer(
    vocab_size=vocab_size,
    embedding_dim=32,
    hidden_dim=32 * 4,
    block_size=block_size,
    dropout_p=0.1
)
print("Refined Model Initialized Successfully!")

Refined Model Initialized Successfully!


In [168]:
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()
epochs = 5

num_train_steps = train_x_mat.shape[1] // block_size

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    # Loop through EVERY batch covering the complete dataset sequentially
    for step in range(num_train_steps):
        xb, yb = get_sequential_batch("train", step)

        logits = model(xb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb.reshape(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / num_train_steps

    # Evaluate on validation set
    model.eval()
    with torch.no_grad():
        val_xb, val_yb = get_sequential_batch("val", 0) # Evaluate on first val chunk
        val_logits = model(val_xb)
        val_loss = loss_fn(val_logits.reshape(-1, vocab_size), val_yb.reshape(-1))
        val_perplexity = torch.exp(val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss.item():.4f} | Val Perplexity: {val_perplexity.item():.4f}")

print("\nTraining on complete dataset complete!")

Epoch [1/5] | Train Loss: 3.2190 | Val Loss: 2.7295 | Val Perplexity: 15.3250
Epoch [2/5] | Train Loss: 2.6787 | Val Loss: 2.5459 | Val Perplexity: 12.7550
Epoch [3/5] | Train Loss: 2.5568 | Val Loss: 2.4437 | Val Perplexity: 11.5159
Epoch [4/5] | Train Loss: 2.4736 | Val Loss: 2.3576 | Val Perplexity: 10.5659
Epoch [5/5] | Train Loss: 2.4157 | Val Loss: 2.3076 | Val Perplexity: 10.0501

Training on complete dataset complete!


In [169]:
model.eval()
# Start with a newline or a prompt character encoded into integers
context = torch.tensor([[stoi['\n']]], dtype=torch.long)

generated_indices = model.generate(context, max_new_tokens=200)[0].tolist()
generated_text = "".join([itos[i] for i in generated_indices])

print("--- Generated Text ---")
print(generated_text)

--- Generated Text ---

KINIUSERLABWhake.
I thek INF hoo Vonor me whimin
CAUchemy th?
I thithin s to indo is spoklo;
Yof me'cetis, feered be utu bato
ON:
Fowand ho twsoour Fhand harte omthy, Af IONGHizlsiy milalemss I swe. F
